<a href="https://colab.research.google.com/github/tmachingur-code/EcoCycle-Hub/blob/main/EcoCycle_Hub_MobileNetV2_Cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EcoCycle Energy Hub : Plastic Material Classification

**Pipeline:** PET · HDPEM · OTHER  
**Model:** MobileNetV2 transfer learning  
**Dataset:** 4,819 validated images in Google Drive

This notebook is the cleaned classification workflow. It does not move, rename, delete, or overwrite source images. YOLO object detection will be added as a separate, verified stage after the classifier is evaluated.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths, classes and reproducibility
The folder name below intentionally includes a space after `Backup` because that is how it appears in the current Google Drive path.

In [ ]:
from pathlib import Path
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

DATASET_PATH = Path(
    "/content/drive/MyDrive/EcoCycle_Hub/01_Original_Dataset_Backup /rawimgs"
)
MODEL_PATH = Path("/content/drive/MyDrive/EcoCycle_Hub/models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["PET", "HDPEM", "OTHER"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_PATH}")

print("Dataset:", DATASET_PATH)
print("Dataset folder exists:", DATASET_PATH.exists())
print("Classes:", CLASS_TO_INDEX)
print("Model output folder:", MODEL_PATH)

## 3. Validate dataset and count images
This cell checks image readability and reports counts. It does not modify the files.

In [ ]:
from PIL import Image

image_files_by_class = {}
dataset_summary = {}

for class_name in CLASS_NAMES:
    class_dir = DATASET_PATH / class_name
    if not class_dir.is_dir():
        raise FileNotFoundError(f"Missing class folder: {class_dir}")

    files = sorted(
        p for p in class_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    valid_files = []
    corrupted_files = []

    for path in files:
        try:
            with Image.open(path) as image:
                image.verify()
            valid_files.append(path)
        except Exception:
            corrupted_files.append(path)

    image_files_by_class[class_name] = valid_files
    dataset_summary[class_name] = {
        "valid": len(valid_files),
        "corrupted": len(corrupted_files)
    }

    print(
        f"{class_name}: {len(valid_files)} valid, "
        f"{len(corrupted_files)} corrupted"
    )

total_valid = sum(len(files) for files in image_files_by_class.values())
print("-" * 45)
print("Total valid images:", total_valid)

if total_valid == 0:
    raise ValueError("No valid images were found.")
if any(dataset_summary[c]["valid"] == 0 for c in CLASS_NAMES):
    raise ValueError("At least one class has no valid images.")

## 4. Visualise sample images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(12, 9))

for row, class_name in enumerate(CLASS_NAMES):
    files = image_files_by_class[class_name]
    sample_count = min(4, len(files))
    samples = random.sample(files, sample_count)

    for col in range(4):
        axes[row, col].axis("off")
        if col < sample_count:
            with Image.open(samples[col]) as image:
                axes[row, col].imshow(image.convert("RGB"))
            axes[row, col].set_title(class_name)

plt.tight_layout()
plt.show()

## 5. Create stratified train, validation and test splits
The split is 70% training, 15% validation and 15% testing. Stratification preserves class proportions in each split.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

records = []
for class_name in CLASS_NAMES:
    for image_path in image_files_by_class[class_name]:
        records.append({
            "image_path": str(image_path),
            "label": class_name
        })

dataset = pd.DataFrame(records)

train_data, temporary_data = train_test_split(
    dataset,
    test_size=0.30,
    stratify=dataset["label"],
    random_state=SEED
)

validation_data, test_data = train_test_split(
    temporary_data,
    test_size=0.50,
    stratify=temporary_data["label"],
    random_state=SEED
)

for split_name, split_df in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Testing", test_data)
]:
    print(f"\n{split_name}: {len(split_df)} images")
    print(split_df["label"].value_counts().reindex(CLASS_NAMES, fill_value=0))

print("\nTotal assigned:", len(train_data) + len(validation_data) + len(test_data))
print("Original total:", len(dataset))

## 6. Prepare TensorFlow datasets
MobileNetV2 preprocessing maps image pixels to the range expected by its pretrained feature extractor.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)

In [ ]:
def load_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = preprocess_input(image)
    return image, label


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    labels = dataframe["label"].map(CLASS_TO_INDEX).to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    dataset = dataset.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_dataset = create_dataset(train_data, training=True)
validation_dataset = create_dataset(validation_data)
test_dataset = create_dataset(test_data)

print("Training, validation and test datasets are ready.")

## 7. Verify one batch

In [ ]:
images, labels = next(iter(train_dataset))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("First five numeric labels:", labels[:5].numpy())
print("Class mapping:", CLASS_TO_INDEX)

## 8. Build the MobileNetV2 classifier
The pretrained ImageNet feature extractor is frozen for the initial training stage.

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(CLASS_NAMES), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 9. Configure class weights and training callbacks
Class weights compensate for the different number of images in each class.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

y_train = train_data["label"].map(CLASS_TO_INDEX).to_numpy()

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(CLASS_NAMES)),
    y=y_train
)
class_weights = {index: float(weight) for index, weight in enumerate(weights)}

BEST_MODEL_FILE = MODEL_PATH / "best_ecocycle_3class_model.keras"

callbacks = [
    ModelCheckpoint(
        filepath=str(BEST_MODEL_FILE),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Class weights:", class_weights)
print("Best model will be saved to:", BEST_MODEL_FILE)

## 10. Train MobileNetV2
This is the main training cell. It may take several minutes depending on the Colab runtime.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

## 11. Plot training history

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 12. Evaluate on the held-out test set
The test set is used only for final evaluation, not for model fitting.

In [ ]:
best_model = tf.keras.models.load_model(BEST_MODEL_FILE)

test_loss, test_accuracy = best_model.evaluate(test_dataset, verbose=1)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

## 13. Classification report and confusion matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true = np.concatenate([batch_labels.numpy() for _, batch_labels in test_dataset])
y_prob = best_model.predict(test_dataset, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    zero_division=0
))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
).plot(cmap="Blues", values_format="d")

plt.title("EcoCycle Energy Hub — Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 14. Review misclassified test images

In [ ]:
misclassified = np.flatnonzero(y_true != y_pred)
print("Incorrect predictions:", len(misclassified))

show_indices = misclassified[:12]

if len(show_indices) == 0:
    print("No misclassified images in this test set.")
else:
    fig, axes = plt.subplots(
        len(show_indices), 1,
        figsize=(8, max(4, len(show_indices) * 3))
    )
    if len(show_indices) == 1:
        axes = [axes]

    for axis, index in zip(axes, show_indices):
        image_path = test_data.iloc[index]["image_path"]
        with Image.open(image_path) as image:
            axis.imshow(image.convert("RGB"))

        axis.set_title(
            f"Actual: {CLASS_NAMES[y_true[index]]} | "
            f"Predicted: {CLASS_NAMES[y_pred[index]]} | "
            f"Confidence: {y_prob[index].max():.1%}"
        )
        axis.axis("off")

    plt.tight_layout()
    plt.show()

## 15. Save and verify the final classifier

In [ ]:
# Save the best classifier in the project models folder.
best_model.save(BEST_MODEL_FILE)

print("Classifier saved:", BEST_MODEL_FILE)
print(f"File size: {BEST_MODEL_FILE.stat().st_size / (1024**2):.2f} MB")

## 16. Next development stage — YOLO object detection

The MobileNetV2 classifier above assigns one material label to an image. It does not locate objects or produce bounding boxes.

The next stage is to prepare a verified YOLO detection dataset, train/evaluate an object detector, and then connect its detected crops to this classifier if the detector and class mapping are suitable.

